# Model & Inquiry: The World Your Question Lives In

**Topic 04 · 1 lecture**

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/honr_464_logo.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>HONR 46400 — Evidence-Driven Research</center>
# <center>Professor: Davi Moreira</center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/notebooks/student/nb04_model_inquiry_student.ipynb)

---

## 🧭 Inquiry & Claim Boundary

**Inquiry emphasis:** the causal kind, caught at the moment you build it. This
notebook constructs the **M**odel and defines the **I**nquiry, the first two
letters of **MIDA**, the course's four-part map of a research design (Model,
Inquiry, Data strategy, Answer strategy). They are the raw material the
**inquiry compass** classifies. The compass sorts any question by **kind**
(descriptive or causal) and **reach** (the data at hand, a population beyond
it, or cases not yet seen). Here you build the thing being sorted. You specify
the world your question lives in and name the exact quantity that answers it.
You also learn to read a formula for its kind: does answering it require a
world that never happened? Get that reading right and the compass call almost
makes itself.

| | |
|---|---|
| **A claim this topic PERMITS** | "Under my stated model of the world, the quantity I want exists and is defined. Here is that quantity, named before I saw any outcome data." |
| **A claim this topic does NOT permit** | "The effect of X" with no world in which X varies (a model you never stated) — and choosing the quantity *after* browsing results, then presenting it as the question you always had. |

**Where this sits in the course:** Phase 2, compass, sources, model, and
measurement. This notebook develops milestone **M03, your Model & Inquiry
Declaration**, which you present as a 3-minute declaration at the Friday
studio. It builds on your Literature & Claim Map (M02): your gap named a
question, and now you specify the world that question assumes and the exact
quantity that would answer it. That is the quantity you first placed on the
compass in nb02.

*Provenance: RDSS ch. 6 'Specifying the model' + ch. 7 'Defining the inquiry' + declaration_7.1 + utilities/make_dag_df.R | ch. 6–7 | possible worlds, potential outcomes, the estimand, a DAG helper, and EDA as model-calibration | translated (R→Python) + newly-constructed-from-source-concept*

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain what a **model** is in the MIDA sense: your written-down picture of
   how the world could work.
2. Read and write **potential outcomes** (Y0 and Y1, the outcomes a person
   would show without and with a treatment) and compute an individual
   treatment effect inside a world you can see completely.
3. State the **fundamental problem of causal inference** in your own words
   after executing it: you never observe both potential outcomes for the same
   unit.
4. Draw a three-node **DAG**, a diagram of what could affect what, and say
   what each arrow claims about the real world.
5. Define an **inquiry**, the exact quantity that would answer your question,
   and tell a descriptive inquiry from a causal one by reading its formula.
6. Declare the model and inquiry for your **own** project (milestone M03).

---

In [ ]:
# Setup — run this cell first.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)
plt.rcParams["figure.figsize"] = (9, 5)

SEED = 464  # course number — keeps every simulation reproducible
rng = np.random.default_rng(SEED)

DATA_URL = ("https://raw.githubusercontent.com/davi-moreira/"
            "2026F_evidence_driven_research_purdue_HONR464/main/notebooks/data/")

def load_course_data(filename):
    """Load a course dataset from GitHub, falling back to the local repo copy."""
    try:
        return pd.read_csv(DATA_URL + filename)
    except Exception:
        from pathlib import Path
        local = Path("notebooks/data") / filename
        if not local.exists():
            local = Path("../data") / filename
        return pd.read_csv(local)

print("✓ Setup complete — seed", SEED)

## 1. Why This Matters

> *"You keep telling me the mentoring program 'works.' Works compared to what?
> Compared to the same undergraduates in a world where they never got a mentor?
> Then show me that world, because that is the only comparison your claim is
> actually about."*
> — a **policy stakeholder** deciding whether to fund the program next year

Every causal claim is secretly a claim about two worlds. There is the world as
it is, and there is a world that did not happen. "The mentor raised her sense
of belonging" means her belonging is higher than it would have been with no
mentor. That second world is one you will never get to see. The invisible
comparison is not a technicality. It is the entire meaning of the word
"effect." Research that cannot say which two worlds it compares is not yet
asking a causal question at all.

MIDA starts here, with its first two letters. The **M**odel is your picture of
what the worlds could look like. The **I**nquiry is the exact quantity you want
from them. In one lecture you will build both. You will construct a small world
you can see *completely*, including the part reality always hides, and then
name the precise number you would ask of it. When the hidden part disappears,
you will feel exactly what makes causal inference hard. You will also know
exactly what you are declaring when you write your own model and inquiry for
milestone M03.

The master map from nb00 returns, the one you were promised you would OWN by
the end of the course. Today you start owning M and I:

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/rdss_fig_5_1.png" width="75%"/></center>

*The MIDA map, the same diagram you met in nb00: a research design has a Model,
an Inquiry, a Data strategy, and an Answer strategy (M·I·D·A), and this
notebook builds the first two. (Figure from the replication materials of Blair,
Coppock & Humphreys (2023),* Research Design in the Social Sciences *,
MIT-licensed archive; the book is free at book.declaredesign.org.)*

> **A question that often comes up here:** *"If the comparison world never
> happens, how can research ever measure an effect?"* That tension is the whole
> game, and it has an honest answer. The answer starts with admitting the
> problem, not hiding it. Today you make the problem vivid. The causal units
> later in the course are all different strategies for reasoning about the
> world you cannot see, each with its own price. You cannot appreciate the
> strategies until you have felt the problem.

---

## 2. The Model: Two Worlds You Can Write Down

**Guiding question:** *when you say a treatment "had an effect," which two
worlds are you secretly comparing, and could you ever stand in both?*

Start with the word **model**. A model is your written-down picture of how the
world could work: which variables exist, what values they can take, and what
could affect what. "Belonging runs 0 to 100, and a mentor can nudge it" is
already a tiny model. Writing the picture down is what turns a hunch into
something the world can contradict.

Next, the vocabulary that makes the two-worlds idea precise. For each person,
imagine two numbers:

- **Y0** is the outcome they *would* have **without** the treatment.
- **Y1** is the outcome they *would* have **with** it.

These are called **potential outcomes**: the two outcomes a person could show
you, depending on whether they get the treatment. Only one of them will
actually occur. The one that does not occur is the **counterfactual**, the
outcome from the world that did not happen. The person's own **individual
treatment effect** is simply Y1 − Y0, how much the treatment moved *their*
outcome.

You will work in one concrete world all course long: the **mentoring-program
world**. Each person is a first-year undergraduate. The outcome is a
sense-of-belonging score from 0 to 100. The treatment is being paired with a
peer mentor. So Y0 is a person's belonging without a mentor, Y1 is their
belonging with one, and Y1 − Y0 is what the mentor did for them.

Here is the trick you get to play, and only a simulation allows it. You will
build a world where you can see **both** Y0 and Y1 for **every** person at
once. Reality never lets you do that. That is precisely why you start in
simulation: to look at the thing reality hides, name it, and then watch it
vanish.

> **A question that often comes up here:** *"Is the mentor's effect the same
> for everyone?"* In reality, almost never. A mentor might lift one person's
> belonging a lot and barely move another's. Real individual effects vary,
> which is why researchers usually end up caring about the *average* effect
> across a group. The teaching world you are about to build deliberately sets
> that variation aside and gives everyone the same effect. The reason will
> become clear the moment both potential-outcome columns are in front of you.

---

### 🔮 Pause & Predict

The next cell simulates 100 people and, because it is a simulation, prints
**both** potential outcomes, Y0 and Y1, side by side for each one.

**Before running it**, commit a prediction. In the *real* mentoring program
(not the simulation), for how many of the 100 people could you ever directly
observe **both** their with-mentor and without-mentor belonging score? Write a
number and one sentence on why.

### YOUR ANSWER HERE:

**Number of people for whom I could observe BOTH potential outcomes in reality:**

**Why:**

---

### 🛠️ Run the Study: build a world you can see completely

Run the cell below. It defines the mentoring-program world and simulates 100
people. You get both potential outcomes and each person's individual treatment
effect, the god's-eye view that only a simulation allows.

> 💡 **Gemini Prompt:** "Here is a Python function from my research-methods
> course. It builds a small simulated 'world' of 100 people, each with two
> potential outcomes: Y0 (belonging without a mentor) and Y1 (belonging with
> one, built as Y0 plus a fixed effect): [paste the next cell]. Walk me through
> what each line does, and explain why being able to see BOTH Y0 and Y1 for the
> same person is something only a simulation can offer."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check Gemini's line-by-line story against the printed table. Do a Y0
>   column, a Y1 column, and an individual_effect column all actually appear?
> - [ ] Confirm its account of the built-in effect: does every person's
>   individual_effect really come out to the same +2.0 the code dialed in?
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# The mentoring-program world. This exact function is reused later in the
# course (nb09-nb11), so read it carefully now.
def make_world(n=100, effect=2.0, noise=2.0, rng=rng):
    """The mentoring-program world: each person's belonging score (0-100)
    without a mentor (Y0) and with one (Y1 = Y0 + effect)."""
    Y0 = rng.normal(50, 10, n) + rng.normal(0, noise, n)
    return pd.DataFrame({"Y0": Y0.round(1), "Y1": (Y0 + effect).round(1)})

world = make_world()
world["individual_effect"] = (world["Y1"] - world["Y0"]).round(1)

print("✓ Simulated a 100-person mentoring-program world (both worlds visible).")
print("\nFirst 8 people — the god's-eye view:")
print(world.head(8).to_string())
print(f"\nIn THIS world, every person's individual effect is the same:"
      f" {world['individual_effect'].iloc[0]} points — by deliberate construction.")

### Reality's move: one world per person

Now do the thing reality does. Each person either gets a mentor or does not.
So for each person, **one** of their two potential outcomes becomes the number
you actually observe. The other silently vanishes into the world that did not
happen. Run the next cell to flip a coin for each person and keep only the
outcome that "really occurred."

The result is the table every real study is stuck with: a column of observed
outcomes, and a ghost column of question marks where the counterfactual used
to be. The individual effect you computed a moment ago becomes uncomputable.
There is no person for whom you still hold both numbers.

> 💡 **Gemini Prompt:** "This cell takes a simulated world where I can see both
> potential outcomes and then does what reality does. It assigns each person to
> mentor or not by a coin flip, keeps only the outcome that matches the
> assignment, and hides the other: [paste the next cell]. Explain what np.where
> is doing on each line, and why the individual_effect column becomes a column
> of question marks afterward."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check that after this cell runs, each row shows a value in EITHER
>   Y0_seen or Y1_seen, never both, matching that person's Z.
> - [ ] Confirm the mentored and not-mentored counts printed below add up to
>   the full 100 people; the coin split need not land at exactly 50/50.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# Reality's move: assign each person to mentor (Z=1) or not (Z=0) by a coin,
# then observe ONLY the potential outcome that matches their assignment.
world["Z"] = rng.integers(0, 2, len(world))                      # the coin
world["observed"] = np.where(world["Z"] == 1, world["Y1"], world["Y0"])

# What a real study actually gets to see: assignment, the observed outcome, and
# a ghost column where the counterfactual is now unknowable.
real_study = world[["Z", "observed"]].copy()
real_study["Y0_seen"] = np.where(world["Z"] == 0, world["Y0"], np.nan)
real_study["Y1_seen"] = np.where(world["Z"] == 1, world["Y1"], np.nan)
real_study["individual_effect"] = "?"   # can never be filled in

print("What a real study is stuck with (first 8 people):")
print(real_study.head(8).to_string())
print(f"\nMentored: {int(world['Z'].sum())} people   "
      f"Not mentored: {int((world['Z'] == 0).sum())} people")
print("\nThe individual_effect column is now unknowable for every single person.")

That missing column is not a data-collection failure you could fix with a
bigger budget or a better survey. It is a logical wall: a person cannot both
have and not have a mentor this semester. This is the **fundamental problem of
causal inference** — for any unit, at most one potential outcome is ever
observable. Every causal method in the rest of this course is, at heart, a
principled way of reasoning about that missing column *without* pretending you
can see it. First, though, you need a way to draw the beliefs that justify any
such reasoning.

---

### 🔍 Reading the Evidence

In the cell below, write two things about the tables you just saw. First: this
world moves *every* person by exactly +2.0 points, a deliberately simple
construction. Name two concrete reasons a real mentoring program would NOT
move everyone equally. Think about who the people are and how a mentor
actually helps. Second, the important one: write the single sentence that
captures *why the both-columns table could never exist for a real mentoring
program*. Be specific about which numbers you could and could not actually
collect.

> **A question that often comes up here:** *"If real effects vary person to
> person, why build a world where they don't?"* Because this world's job is to
> be an **answer key**. In the coming units you will hand this world to
> estimators and simulations and ask how close they land to the truth. For
> that, the truth needs to be a single crisp number. In a richer world, the
> realized average effect for these particular 100 people would drift away
> from the +2.0 dial, since each sample of people carries its own average. You
> will meet exactly that idea when nb10 takes on generalization. Simple first,
> rich later, on purpose.

### YOUR ANSWER HERE:

**Two reasons a real program's effects would vary from person to person:**

**Why the both-columns table could never exist for a real program:**

---

## 3. Drawing the Model: DAGs

**Guiding question:** *how do you put a belief about what causes what on
paper, clearly enough that the world could prove you wrong?*

So far your model lives in prose. The standard way to draw it is a **DAG**
(directed acyclic graph). Do not let the name intimidate you; a DAG is a drawn
story about what could affect what, and you need no graph theory to read one.
Each **node** is a circle standing for a variable, like "peer mentor" or
"belonging." Each **arrow** is a claim that one variable *could* directly
influence another. "Acyclic" just means no variable ends up causing itself
through a loop. The smallest honest picture has two nodes and one arrow, a
treatment Z pointing at an outcome Y, plus a U acknowledging that units differ
in ways you have not named:

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/rdss_fig_6_1.png" width="55%"/></center>

*A minimal DAG: the treatment Z points at the outcome Y, with U standing for
the unknown ways people differ. (Figure from the replication materials of
Blair, Coppock & Humphreys (2023),* Research Design in the Social Sciences *,
MIT-licensed archive; the book is free at book.declaredesign.org.)*

Why does a picture earn its place? The moment you draw the arrows, you commit
to a story the world could confirm or contradict. You also expose the
variables that quietly threaten your causal claim. In the mentoring world,
suppose people who were **already more involved on campus** are both more
likely to seek out a mentor *and* more likely to feel they belong anyway. Then
prior involvement is a **confounder**: a common cause of both the treatment
and the outcome. Any naive comparison of mentored versus unmentored people is
then partly measuring involvement, not the mentor. A DAG makes that danger
visible as a shape you can point at. Run the next cell to draw exactly this
three-node model.

> 💡 **Gemini Prompt:** "Here is Python that draws a three-node DAG from an
> edge list using networkx. The nodes are prior involvement, peer mentor, and
> belonging: [paste the next cell]. Explain what a directed edge asserts about
> the world, and tell me in plain terms why 'prior involvement' pointing at
> BOTH 'peer mentor' and 'belonging' is what makes it a confounder."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check the drawn graph against the edges list: are there exactly three
>   arrows, and do two of them start at 'prior involvement'?
> - [ ] Confirm Gemini's confounder explanation matches the picture. A
>   confounder has an arrow into the treatment AND an arrow into the outcome.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# A model of the mentoring world, drawn from an edge list. Each arrow is a
# belief: "this variable could directly affect that one."
import networkx as nx

edges = [
    ("prior involvement", "peer mentor"),   # already-involved people seek mentors
    ("prior involvement", "belonging"),     # ...and tend to feel they belong anyway
    ("peer mentor",       "belonging"),      # the causal arrow we actually care about
]
dag = nx.DiGraph(edges)

pos = nx.spring_layout(dag, seed=464)   # seed keeps the layout reproducible
fig, ax = plt.subplots(figsize=(8, 5))
nx.draw_networkx(dag, pos=pos, ax=ax, node_size=3000, node_color="#dbe4ff",
                 edgecolors="#3b5bdb", linewidths=1.5, font_size=10,
                 arrowsize=22, width=1.5)
ax.set_title("A three-node model of the mentoring world "
             "(prior involvement confounds mentor → belonging)")
ax.set_axis_off()
fig.tight_layout()
plt.show()

print("Nodes:", list(dag.nodes()))
print("Arrows (each is a claim about the real world):", list(dag.edges()))

### ⚖️ Make a Design Choice

Your DAG is an argument, and every arrow is contestable. In the cell below, do
two things. First, name **one more** variable that could plausibly be a
confounder in the mentoring world, a common cause of both getting a mentor and
feeling belonging. Decide whether to add it to the model, and defend the call
in a sentence. Second, pick the single arrow in your DAG you are **least**
sure about and state what real-world observation would make you erase it.

> 💡 **Gemini Prompt:** "I am modeling whether [your treatment] affects [your
> outcome] for [your units]. List variables that could be common causes of
> both, the potential confounders I should consider putting in my DAG. For
> each one, state the mechanism by which it would influence both."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] For each suggested confounder, decide *yourself* whether the mechanism
>   is real for your context. The AI is brainstorming candidates, not settling
>   facts.
> - [ ] Confirm each is genuinely a *common cause* (points to both treatment
>   and outcome), not just something correlated. Draw the arrows and check.
> - [ ] Log this use in your AI ledger: tool, task, verification.

### YOUR ANSWER HERE:

**One more candidate confounder + whether I add it (and why):**

**The arrow I trust least + what observation would make me erase it:**

---

### 🎯 Project Transfer — draw your project's DAG

Adapt the DAG cell above to your **own** project. Replace the three node
labels with your treatment (or main explanatory variable), your outcome, and
one lurking third variable that could affect both. Copy the code into the cell
below, edit the `edges` list, and run it. Then read the picture back to
yourself. Does every arrow assert something you actually believe about the
world? Pick the arrow you care most about and say, out loud, the real-world
mechanism it asserts: the actual story by which the first variable could move
the second.

### YOUR SOLUTION HERE
# Paste the DAG code from above and edit the three node labels + edges list
# to match your own project's world. Then run it.


## 4. Ground the Model in Real Data

**Guiding question:** *your model needs plausible numbers; where do they come
from?*

You just built a model out of thin air: belonging runs 0 to 100, the mentor
adds a tidy +2. That is fine for a teaching world. For *your* project, the
numbers have to answer to reality. The move that grounds them is
**exploratory data analysis (EDA)** — a quick, no-commitment look at real data
*before* you declare anything. Two things come out of that look. Your
**M**odel gets real-world ranges and shapes to imitate instead of invented
ones. And candidate **I**nquiries, the quantities actually worth chasing,
surface from patterns you can see.

To feel the move, borrow the real LAPOP Brazil survey (you will live inside it
for weeks starting in nb06) and take a two-minute look. What range do its
trust items occupy? What shape does one of them take? Do two of them move
together?

> 💡 **Gemini Prompt:** "Here is a short exploratory pass over a real survey
> dataset. It prints the range and mean of two 1–7 trust items, the full
> distribution of a third, and the correlation between two of them: [paste the
> next cell]. Explain what each piece tells me about the *shape* of real data,
> and why looking at this BEFORE I fix the plausible values in a model keeps
> the model honest."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check Gemini's read of the correlation's sign and size against the
>   number the cell prints. A positive value near 0.5 means the two items tend
>   to rise together, loosely, not in lockstep.
> - [ ] Confirm nothing here answers a research question yet. The cell only
>   *describes* what is in front of it; catch Gemini if it slips into "so X
>   causes Y" or "so this holds for all Brazilians."
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# EDA upstream: a two-minute look at REAL data before declaring anything.
# (No rng here — exploration only describes what is already in the file.)
lapop = load_course_data("lapop_brazil.csv")
assert lapop.shape == (10000, 10), "unexpected shape — flag this!"

tp = lapop["trust_police"]
tm = lapop["trust_military"]

print("Real-world ranges and centers (what a Model of these should imitate):")
print(f"  trust_police   : {tp.min()}-{tp.max()} on the scale, mean {tp.mean():.2f}")
print(f"  trust_military : {tm.min()}-{tm.max()} on the scale, mean {tm.mean():.2f}")

print("\nShape of one item — trust in the supreme court, counts per scale point:")
print(lapop["trust_supreme_court"].value_counts().sort_index().to_string())

corr = tp.corr(tm)
print(f"\nDo two items move together? corr(trust_police, trust_military) = {corr:.2f}")
print("  positive and moderate: people who trust one institution tend to trust")
print("  the other — a *candidate* inquiry worth declaring, not an answer yet.")

In [ ]:
# Self-check — these are facts about the file, not the simulation (no seed needed).
assert round(tp.mean(), 2) == 4.43 and round(tm.mean(), 2) == 5.16, "means drifted — re-load"
assert round(corr, 2) == 0.46, "correlation drifted — investigate before trusting it"
print("✓ Self-check passed: trust_police mean 4.43, trust_military mean 5.16, and the")
print("  two items correlate about 0.46 across these 10,000 interviews.")

### What just happened, and what did not

Read what you did carefully, because it is easy to mistake for research. You
did **not** answer a question. You did not show that trusting the police
*makes* people trust the military, and you did not learn what "most
Brazilians" think. What you did was two upstream jobs. You **calibrated M**:
real trust items run 1 to 7, pile up unevenly, and lean high, so a model of
them should borrow those ranges and that shape rather than a tidy invented
bell curve. And you **surfaced candidate I's**: that \~0.46 correlation is a
*lead*, not a finding. It is a quantity ("how tightly do trust items move
together?") that might be worth declaring as an inquiry and chasing properly.

Here is the discipline, and much of the course turns on it: **an exploration
is not an answer.** A pattern you spot while looking around becomes a claim
you are allowed to make only *after* you declare it as a precise inquiry, and
ideally after you check it against data you did not explore. That **explore →
declare → confirm** loop gets its name and its hands-on drill in nb06. For
now, let exploration do its upstream job. It feeds your model real shapes, and
it hands you candidate inquiries to carry into the genie test next.

---

## 5. The Inquiry: The Genie Test

> *"Before you collect a single data point, tell me the exact number you are
> trying to learn. If a genie appeared and offered you the truth, what would
> you ask it for? If you cannot say, you are not ready to gather data. You are
> ready to go fishing."*
> — a **thesis advisor**, saving you from a semester of aimless analysis

The **inquiry** is that number: the precise quantity a genie with perfect
knowledge would hand you. Researchers also call it the **estimand**, which
just means "the thing to be estimated." It is the I in MIDA, and defining it
is a separate act from building the model. The model says which worlds exist
and how they work. The inquiry says which single number you want from them. In
the mentoring world, the most common inquiry is the **Average Treatment Effect
(ATE)**: the average of every person's individual effect across the whole
world.

**ATE = average of (Y1 − Y0) over all units.**

Because you built this world in a simulation, you sit in the genie's seat. You
hold every Y0 and every Y1, so you can compute the ATE exactly. Real studies
never can. That is exactly why the inquiry has to be *declared*, in words and
as a formula, before the data arrive. Declaring it first is what stops you
from quietly redefining "the effect" later to match whatever your results
happened to show.

> **A question that often comes up here:** *"Why write a formula when I can
> just say 'the effect'?"* Because "the effect" is ambiguous and a formula is
> not. The average effect over everyone, the effect just for people who chose
> the mentor, the share of people helped at all: these are different numbers
> with different formulas, and they can point in different directions. The
> formula forces you to pick one *before* the data can tempt you toward
> whichever looks best.

---

### 🔮 Pause & Predict

Look back at the `make_world` function. It builds `Y1 = Y0 + effect`, with
`effect=2.0`, a nudge of +2 points wired into the machine.

**Before running the next cell**, predict: when you compute the *actual* ATE,
the average of the 100 individual effects in this specific world, will it come
out **exactly 2.0**, **a little off from 2.0**, or **far from 2.0**? One
sentence on why. This is a read-the-machine question; the answer is sitting in
the line that builds Y1.

### YOUR ANSWER HERE:

**The computed ATE will be (exactly 2.0 / a little off / far off):**

**Why:**

---

> 💡 **Gemini Prompt:** "This cell computes the 'true' average treatment effect
> in a simulated world by averaging Y1 minus Y0 over all 100 people, then
> argues the number becomes hidden once you keep only one outcome per person:
> [paste the next cell]. Explain why the average comes out to exactly the
> effect the world was built with, and what the phrase 'the ATE is the answer
> key' means here."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check the printed TRUE ATE against the +2.0 the world dialed in. The
>   assert line guarantees the match, so confirm the value really prints as 2.0.
> - [ ] Make sure you can say WHY it is exact (every person gets the same
>   effect), not just that it is. Gemini should point you to the
>   Y1 = Y0 + effect line.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# We are the genie: compute the exact inquiry value for THIS world.
TRUE_ATE = (world["Y1"] - world["Y0"]).mean()

print(f"The dialed-in nudge was +2.0 points per person.")
print(f"The TRUE ATE realized in these 100 people is: {TRUE_ATE:.3f}")

# Self-check: constant-effect construction means the ATE equals the dial exactly.
assert abs(TRUE_ATE - 2.0) < 0.01, "constant-effect world must realize ATE = 2.0"
print("\n✓ This number is the ANSWER KEY.")
print("We now 'hide' it: from the observed (one-outcome-per-person) data alone,")
print("no one can recompute it — later units will try to RECOVER it with estimators.")

# Prove it is hidden: the observed data cannot rebuild the individual effects.
can_compute_from_observed = world[["Z", "observed"]].assign(effect="unknowable")
print("\nFrom observed data alone, every individual effect stays:",
      can_compute_from_observed["effect"].unique()[0])

The ATE came out exactly 2.0, and the machine guaranteed it: the
`Y1 = Y0 + effect` line hands every person the same nudge, so the average of
100 identical effects is the dial itself. In a richer world where effects vary
person to person, the average in one sample of 100 people would drift away
from the average in the whole population they came from. Hold that thought;
nb10 is built on it.

---

## 6. Reading the Kind from the Formula

**Guiding question:** *how can you tell, from the formula alone, whether your
question needs a world that never happened?*

Not every inquiry is a treatment effect. The genie test works for any
quantity; the question is just *which* quantity you ask for. Two broad
families:

- A **descriptive inquiry** asks about the world **as it is**. No second world
  required. "What is the average belonging score?" is the mean of Y. "What
  share of first-years score above 60?" is a proportion. These need only the
  outcomes you can, in principle, observe.
- A **causal inquiry** asks about the **difference between two worlds**, so it
  needs potential outcomes. The ATE is the flagship example: the average of
  Y1 − Y0.

The tell is whether the quantity *requires a world that did not happen*. A
description never does; a causal inquiry always does. That single distinction
fixes your question's **kind**, the first axis of the inquiry compass, and
with it how hard the answer will be to earn and what could go wrong on the
way. Writing your inquiry as a formula makes the kind obvious. If a
counterfactual (Y0 or Y1 for a unit that did not get that condition) appears
in the formula, you are making a causal claim and owe the world that justifies
it.

---

### 📝 Practice

For each inquiry below, (i) write it as a **formula** over outcomes or
potential outcomes, and (ii) label it **descriptive** or **causal**.

- **A.** "The average sense-of-belonging score among all first-year
  undergraduates."
- **B.** "The average effect of the peer-mentor program on belonging."
- **C.** "The share of first-years who feel they belong (score above 60)."
- **D.** "How much higher a person's belonging is with a mentor than it would
  have been without one, averaged over people."


### YOUR ANSWER HERE:

**A – formula / type:**

**B – formula / type:**

**C – formula / type:**

**D – formula / type:**

---

## 7. How M and I Pin Each Other Down

The two letters are not separate chores; each one disciplines the other.
Without a model, your inquiry is homeless. "The effect of mentoring" names no
world in which mentoring varies, so there is no number to ask the genie for.
Without an inquiry, your model is aimless. A DAG full of arrows with no stated
quantity invites you to go fishing and call whatever bites "the finding."
Write them together and each exposes the other's gaps. If your inquiry
mentions a variable your DAG does not contain, the model is missing a piece.
If your DAG flags a confounder, your inquiry's causal formula tells you
exactly why that confounder matters: it corrupts the comparison between the
two worlds your formula names. This is why milestone M03 asks for both in one
declaration.

---

### 🎯 Project Transfer — declare your model and inquiry (M03)

This is the heart of milestone M03, which you present as a 3-minute
declaration at the Friday studio. In the cell below, assemble it: (1) your
**model** in one or two sentences, meaning the world your question assumes
plus the one confounder your DAG flagged, and (2) your **inquiry** stated the
genie way, in one plain-English sentence *and* a formula, labeled descriptive
or causal. Declaring it now, before your data strategy, is the discipline that
protects the rest of your project from inquiry-shopping.

### YOUR ANSWER HERE:

**My model (the world + the key confounder from my DAG):**

**My inquiry, in plain English (the genie sentence):**

**My inquiry, as a formula:**

**Descriptive or causal (and how the formula tells me):**

---

## 8. If You Want to Go Further *(optional)*

Nothing in this section is required before the Friday studio, and none of it
will be covered in lecture. It deepens what you just did, and you can read it
in about ten minutes.

**The full zoo of arrow patterns.** A confounder is only the first of a small
family of shapes, and each one carries a different threat (or opportunity) for
a causal claim:

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/rdss_fig_6_2.png" width="80%"/></center>

*The vocabulary of arrow patterns: a confounder, a moderator, a mediator, a
collider, and an instrument, five roles a third variable can play between a
treatment and an outcome. (Figure from the replication materials of Blair,
Coppock & Humphreys (2023),* Research Design in the Social Sciences *,
MIT-licensed archive; the book is free at book.declaredesign.org.)*

You will meet these shapes again in the causal units. One warning is worth
carrying now: not every third variable should be added to your model as
something to "control for." A **collider** (a variable caused by *both* the
treatment and the outcome) actually creates a false association when you
adjust for it. Drawing the DAG first is what saves you from that trap.

**Ask the genie for a different number.** The ATE is one inquiry among many,
and the genie will answer any well-posed question. Try writing formulas for
two variations. The *average effect among the treated* asks: among the people
who actually got mentors, what was the average of Y1 − Y0? The *share helped*
asks: for what fraction of people is Y1 greater than Y0? Both are legitimate
inquiries, both need potential outcomes, and both can differ from the ATE in
the very same world. If you can write each as a formula and say exactly who it
averages over, the estimand idea is fully yours.

### 📝 Practice

For each question below, name the **potential-outcomes pair** it implicitly
compares, the "with" world and the "without" world for the unit. Write each as
"Y1 = … ; Y0 = …".

- **A.** "Does a text-message reminder increase the chance a voter turns out?"
- **B.** "Does switching a course to pass/fail change how many hours
  first-years study for it?"
- **C.** "Did the new campus shuttle reduce a commuter's average travel time?"


### YOUR ANSWER HERE:

**A – Y1 / Y0:**

**B – Y1 / Y0:**

**C – Y1 / Y0:**

---

### 🎟️ Claim Ticket

**Claim Ticket #1** — in the cell below, state your inquiry in **one sentence
a non-researcher could repeat back to you**, with no formula and no jargon.
Then name **one arrow** in your DAG and the real-world mechanism it asserts.
If a friend could not restate your inquiry, it is not ready.

### YOUR ANSWER HERE:

**My inquiry, one plain sentence:**

**One arrow in my DAG + the real-world mechanism it asserts:**

---

## 9. Wrap-Up

In one lecture you assembled the first half of MIDA. You learned that every
causal claim compares two worlds, and you wrote those worlds down as potential
outcomes. You executed the fundamental problem of causal inference by watching
one potential outcome per person vanish into the counterfactual you can never
see. You drew a model as a DAG, where every arrow is a contestable claim and a
common cause is a visible threat, and you grounded a model's numbers in real
data with a two-minute exploration. Finally, you defined an inquiry with the
genie test: the exact number, in words and a formula, declared *before* the
data can tempt you. Along the way you learned to tell a description from a
causal effect by asking whether the formula needs a world that did not happen.

> **"A causal question you cannot draw as two worlds and write as one formula
> is not yet a question — it is a wish. The model says which worlds; the
> inquiry says which number; only then are you allowed to go looking."**

Bring your M03 declaration to the Friday studio. You present it in three
minutes, and it is the foundation everything after it stands on. Next, nb05
takes the quantity you just declared and asks the following hard question: how
do you turn the abstract construct in your inquiry into a number you can
actually collect, and how wrong can that number be before your answer changes?

---

## 10. Sources & Provenance

**Provenance of this notebook:**
- *RDSS ch. 6 'Specifying the model' | the M of MIDA | possible worlds + potential outcomes Y0/Y1 | translated (R→Python, honors non-quant framing)*
- *RDSS ch. 7 'Defining the inquiry' + declaration_7.1 | the I of MIDA | the estimand / ATE and the descriptive-vs-causal (kind) split | translated (R→Python)*
- *utilities/make_dag_df.R | DAG helper | edge-list DAG drawn with networkx spring_layout (seed 464) | translated (R→Python)*
- *RDSS ch. 6–7 + lapop_brazil.csv | EDA upstream of declaration | a two-minute exploratory pass that calibrates M (real ranges/shapes) and surfaces candidate inquiries; the explore→declare→confirm loop is named and anchored in nb06 | newly-constructed-from-source-concept*
- *fresh | the mentoring-program world simulation (seed 464) + the genie-test framing | — | newly-constructed-from-source-concept*

**Readings:**
- Blair, G., Coppock, A., & Humphreys, M. (2023). *Research Design in the
  Social Sciences*, ch. 6–7. Free online:
  [book.declaredesign.org](https://book.declaredesign.org/).

---

<center>

Thank you!

</center>